In [18]:
!pip install pypdf tiktoken langchain-text-splitters google-genai chromadb ipywidgets tqdm python-dotenv -q

# Beca 18 RAG Chatbot
## Pipeline Overview
This notebook implements an end-to-end Retrieval-Augmented Generation (RAG)
pipeline to answer questions about the official Beca 18 regulations (PRONABEC 2026).

**Pipeline:** PDF → Text Extraction → Chunking → Embeddings → ChromaDB → Semantic Search → Grounded Generation → Chat UI

In [3]:
# Step 0 — Environment Setup
import pypdf
import tiktoken
import chromadb
import google.genai
import langchain_text_splitters
import ipywidgets
import tqdm
import dotenv

print("📦 Library Versions:")
print(f"  pypdf                   : {pypdf.__version__}")
print(f"  tiktoken                : {tiktoken.__version__}")
print(f"  chromadb                : {chromadb.__version__}")
print(f"  google-genai            : {google.genai.__version__}")
print(f"  langchain-text-splitters: OK (no __version__)")
print(f"  ipywidgets              : {ipywidgets.__version__}")
print(f"  tqdm                    : {tqdm.__version__}")

print("\n✅ Environment ready!")

📦 Library Versions:
  pypdf                   : 6.11.0
  tiktoken                : 0.12.0
  chromadb                : 1.5.9
  google-genai            : 1.68.0
  langchain-text-splitters: OK (no __version__)
  ipywidgets              : 7.7.1
  tqdm                    : 4.67.3

✅ Environment ready!


In [19]:
# Step 0b — Load API Key
import os
from google.colab import userdata

# Cargar desde Colab Secrets (más seguro que .env en Colab)
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

from google import genai
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

print("✅ Gemini API key loaded successfully!")

✅ Gemini API key loaded successfully!


In [20]:
# Step 1 — PDF Upload
from google.colab import files
uploaded = files.upload()  # Sube beca18_reglamento.pdf

Saving 7778068-rde-n-033-2026-minedu-vmgi-pronabec.pdf to 7778068-rde-n-033-2026-minedu-vmgi-pronabec (1).pdf


In [21]:
import os, re, time, random
import pypdf, tiktoken, chromadb
import google.genai as genai
from google.colab import userdata
from langchain_text_splitters import RecursiveCharacterTextSplitter
from tqdm import tqdm

os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
print("✅ Listo!")

✅ Listo!


In [22]:
PDF_PATH = "7778068-rde-n-033-2026-minedu-vmgi-pronabec.pdf"

def extract_text_from_pdf(path):
    pages_text = []
    with open(path, "rb") as f:
        reader = pypdf.PdfReader(f)
        for i, page in enumerate(reader.pages):
            raw = page.extract_text() or ""
            raw = re.sub(r' +', ' ', raw)
            raw = re.sub(r'\n{3,}', '\n\n', raw)
            pages_text.append(f"[PAGE {i+1}]\n{raw.strip()}")
    return "\n\n".join(pages_text)

full_text = extract_text_from_pdf(PDF_PATH)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=400, chunk_overlap=60,
    separators=["\n\n", "\n", ". ", " "]
)
chunks = []
for i, text in enumerate(splitter.split_text(full_text)):
    chunks.append({
        "id": f"chunk_{i:04d}",
        "text": text,
        "metadata": {"document": "RDE-033-2026-PRONABEC", "topic": "Beca 18 Reglamento", "language": "es"}
    })
print(f"✅ {len(chunks)} chunks listos!")

✅ 1172 chunks listos!


## Step 2 — Tokenization and Chunking Justification
The document contains 109,870 tokens across 138 pages, far exceeding
the 8,192-token embedding limit of gemini-embedding-001.

A chunk size of 400 tokens with 60-token overlap was chosen because:
- 400 tokens ≈ 2–3 paragraphs, enough to capture one complete regulatory idea
- 60-token overlap ensures sentences split across boundaries are still retrievable
- RecursiveCharacterTextSplitter respects natural paragraph structure

In [23]:
# Buscar más específico
keywords2 = [
    "subvencion mensual",
    "monto mensual",
    "pension mensual",
    "obligaciones del becario",
    "obligaciones de los becarios",
    "causales de suspension",
    "causales de cancelacion",
    "perdida de la beca",
    "S/ ",
    "soles mensuales"
]

for kw in keywords2:
    idx = full_text.lower().find(kw.lower())
    if idx != -1:
        text_before = full_text[:idx]
        last_page = re.findall(r'\[PAGE (\d+)\]', text_before)
        page_num = last_page[-1] if last_page else "?"
        print(f"\n'{kw}' → PAGE {page_num}")
        print(full_text[idx:idx+300])
        print("...")


'obligaciones del becario' → PAGE 122
obligaciones del becario, se encuentra el cumplimiento 
del “Compromiso del Servicio al Perú”, a fin de revertir a favor del país los 
beneficios de la capacitación recibida por el Estado (Anexo N° 08), 
conforme a lo estipulado en la Ley N° 29837 y su Reglamento. 
 
 
Esta es una copia autenticada 
...

'obligaciones de los becarios' → PAGE 122
obligaciones de los becarios 
 
19.1. Los derechos y obligaciones de los becarios se rigen de acuerdo con la 
Ley N° 29837 y su Reglamento y a las normas internas del Programa. 
 
19.2. Todos los beneficios para becarios establecidos en el artículo 8 de las 
presentes bases se hacen efectivos a part
...

'S/ ' → PAGE 4
S/ 100,000,000.00 (Cien millones con 00/100 soles) en el clasificador de gasto 
2.5.3.1.1.1 a estudiantes por la fuente de financiamiento de Recursos Ordinarios, según 
la programación nominal elaborada y remitida por la D IBEC; finalmente, la Unidad de 
Estudios Sociales e Investigación de 

In [24]:
# Buscar artículos clave
keywords4 = [
    "articulo 8",
    "art. 8",
    "beneficios de la beca",
    "subvenci",
    "pensi",
    "retiro",
    "renuncia",
    "incumplimiento",
    "desaprueba",
    "promedio"
]

for kw in keywords4:
    # Buscar TODAS las ocurrencias
    for m in re.finditer(kw.lower(), full_text.lower()):
        idx = m.start()
        text_before = full_text[:idx]
        last_page = re.findall(r'\[PAGE (\d+)\]', text_before)
        page_num = last_page[-1] if last_page else "?"
        if int(page_num) > 50:  # solo páginas después de 50
            print(f"'{kw}' → PAGE {page_num}: {full_text[idx:idx+150]}")
            print()
            break

'beneficios de la beca' → PAGE 136: beneficios de la beca, 
debiendo, en caso de desaprobar un curso o más, asumir las 
responsabilidades administrativas de acuerdo a la normatividad vig

'subvenci' → PAGE 92: subvencionan los costos académicos, no 
académicos y/o administrativos de la misma, según corresponda. 
 
8.2. De acuerdo con la normatividad vigente,

'pensi' → PAGE 92: Pensión de estudios 
Nivelación académica 
Dentro del primer ciclo de estudios, siempre y 
cuando sea parte integral de la Malla Curricular o 
Plan de

'retiro' → PAGE 111: retiro total del 
programa de estudios. 
 
13.1.2 Haber sido adjudicado con una beca que subvencione el Estado 
para el mismo nivel de estudios (pregr

'renuncia' → PAGE 111: renuncia y pérdida de la 
beca o crédito. 
 
13.1.3 Haber renunciado a una beca o crédito que canalice, gestione y/o 
subvencione el PRONABEC, para el

'incumplimiento' → PAGE 123: incumplimiento de cualquiera de las estipulaciones, condiciones 
y/o requisitos establecid

In [25]:
# Extraer texto de páginas clave y agregar a ChromaDB
key_pages = [8, 92, 111, 122, 123, 124, 125]

extra_chunks = []
for page_num in key_pages:
    # Encontrar el texto de esa página
    pattern = f"[PAGE {page_num}]"
    idx = full_text.find(pattern)
    if idx != -1:
        next_page = full_text.find(f"[PAGE {page_num+1}]", idx)
        if next_page == -1:
            page_text = full_text[idx:idx+2000]
        else:
            page_text = full_text[idx:next_page]

        # Dividir en chunks
        page_chunks = splitter.split_text(page_text)
        for j, text in enumerate(page_chunks):
            extra_chunks.append({
                "id": f"extra_p{page_num}_{j:02d}",
                "text": text,
                "metadata": {"document": "RDE-033-2026-PRONABEC", "topic": "Beca 18 Reglamento", "language": "es"}
            })

print(f"Extra chunks generados: {len(extra_chunks)}")
for c in extra_chunks[:3]:
    print(f"\n{c['id']}: {c['text'][:100]}...")

Extra chunks generados: 48

extra_p8_00: [PAGE 8]
2 
 
9. Recursos financieros ................................ ................................

extra_p8_01: 11.1. Detalle explicativo para la conversión o equivalencia numérica de las 
calificaciones literale...

extra_p8_02: 11.1.1 Sobre las competencias consideradas para el cálculo del rendimiento 
académico de Beca 18 y B...


## Embedding Strategy
We use `gemini-embedding-001` with two task types:
- `RETRIEVAL_DOCUMENT` for indexing chunks into ChromaDB
- `RETRIEVAL_QUERY` for embedding user questions at query time

Exponential backoff handles the free-tier rate limit (~60 requests/minute).
Dimensions: 3072 (actual output of gemini-embedding-001).

In [26]:
# Funciones de embedding
def embed_query(text, client, max_retries=5):
    for attempt in range(max_retries):
        try:
            result = client.models.embed_content(
                model="gemini-embedding-001",
                contents=text,
                config={"task_type": "RETRIEVAL_QUERY"}
            )
            return result.embeddings[0].values
        except Exception as e:
            if attempt == max_retries - 1:
                raise e
            time.sleep((2 ** attempt) + random.uniform(0, 1))

def embed_documents(texts, client, max_retries=5):
    embeddings = []
    for i, text in enumerate(tqdm(texts, desc="Embedding")):
        for attempt in range(max_retries):
            try:
                result = client.models.embed_content(
                    model="gemini-embedding-001",
                    contents=text,
                    config={"task_type": "RETRIEVAL_DOCUMENT"}
                )
                embeddings.append(result.embeddings[0].values)
                time.sleep(1.1)
                break
            except Exception as e:
                if attempt == max_retries - 1:
                    embeddings.append([0.0] * 3072)
                else:
                    time.sleep((2 ** attempt) + random.uniform(0, 1))
    return embeddings

# Cargar ChromaDB existente
chroma_client = chromadb.PersistentClient(path="chroma_db_beca18")
COLLECTION_NAME = "beca18_chunks"
collection = chroma_client.get_collection(COLLECTION_NAME)
print(f"✅ ChromaDB cargado: {collection.count()} documentos")

NotFoundError: Collection [beca18_chunks] does not exist

## Step 4 — Vector Database Design
- Distance metric: Cosine similarity (best for semantic text search)
- Idempotent indexing: checks if collection exists before embedding
- Persistent storage: chroma_db_beca18/ (excluded from git)
- Collection stores: document text, metadata, and 3072-dim embeddings

In [27]:
# CELDA ÚNICA — Setup completo + indexación de chunks clave
import os, re, time, random
import pypdf, chromadb
import google.genai as genai
from google.colab import userdata
from langchain_text_splitters import RecursiveCharacterTextSplitter
from tqdm import tqdm

# API key
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

# Funciones embedding
def embed_query(text, client, max_retries=5):
    for attempt in range(max_retries):
        try:
            result = client.models.embed_content(
                model="gemini-embedding-001", contents=text,
                config={"task_type": "RETRIEVAL_QUERY"})
            return result.embeddings[0].values
        except Exception as e:
            if attempt == max_retries - 1: raise e
            time.sleep((2**attempt) + random.uniform(0,1))

def embed_documents(texts, client, max_retries=5):
    embeddings = []
    for i, text in enumerate(tqdm(texts, desc="Embedding")):
        for attempt in range(max_retries):
            try:
                result = client.models.embed_content(
                    model="gemini-embedding-001", contents=text,
                    config={"task_type": "RETRIEVAL_DOCUMENT"})
                embeddings.append(result.embeddings[0].values)
                time.sleep(1.1)
                break
            except Exception as e:
                if attempt == max_retries - 1:
                    embeddings.append([0.0]*3072)
                else:
                    time.sleep((2**attempt) + random.uniform(0,1))
    return embeddings

# Extraer texto
PDF_PATH = "7778068-rde-n-033-2026-minedu-vmgi-pronabec.pdf"
pages_text = []
with open(PDF_PATH, "rb") as f:
    reader = pypdf.PdfReader(f)
    for i, page in enumerate(reader.pages):
        raw = page.extract_text() or ""
        raw = re.sub(r' +', ' ', raw)
        raw = re.sub(r'\n{3,}', '\n\n', raw)
        pages_text.append(f"[PAGE {i+1}]\n{raw.strip()}")
full_text = "\n\n".join(pages_text)
print(f"PDF extraido: {len(pages_text)} paginas")

# Chunks de páginas clave solamente (para no agotar el límite)
key_pages = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10,
             90, 91, 92, 93, 94, 95,
             110, 111, 112, 113,
             120, 121, 122, 123, 124, 125, 126,
             130, 131, 132, 133, 134, 135, 136]

splitter = RecursiveCharacterTextSplitter(
    chunk_size=400, chunk_overlap=60,
    separators=["\n\n", "\n", ". ", " "]
)

chunks = []
for page_num in key_pages:
    pattern = f"[PAGE {page_num}]"
    idx = full_text.find(pattern)
    if idx != -1:
        next_page = full_text.find(f"[PAGE {page_num+1}]", idx)
        page_text = full_text[idx:next_page] if next_page != -1 else full_text[idx:idx+2000]
        for j, text in enumerate(splitter.split_text(page_text)):
            chunks.append({
                "id": f"p{page_num}_{j:02d}",
                "text": text,
                "metadata": {"document": "RDE-033-2026-PRONABEC",
                             "topic": "Beca 18 Reglamento", "language": "es"}
            })

print(f"Chunks de paginas clave: {len(chunks)}")

# ChromaDB
chroma_client = chromadb.PersistentClient(path="chroma_db_beca18")
COLLECTION_NAME = "beca18_chunks"
existing = [c.name for c in chroma_client.list_collections()]
if COLLECTION_NAME in existing:
    chroma_client.delete_collection(COLLECTION_NAME)
collection = chroma_client.create_collection(
    name=COLLECTION_NAME, metadata={"hnsw:space": "cosine"})

# Indexar
print(f"\nIndexando {len(chunks)} chunks...")
embeddings = embed_documents([c["text"] for c in chunks], client)
collection.add(
    ids       = [c["id"]       for c in chunks],
    documents = [c["text"]     for c in chunks],
    embeddings= embeddings,
    metadatas = [c["metadata"] for c in chunks]
)
print(f"\nTotal documentos: {collection.count()}")

PDF extraido: 138 paginas
Chunks de paginas clave: 280

Indexando 280 chunks...


Embedding: 100%|██████████| 280/280 [1:20:36<00:00, 17.27s/it]



Total documentos: 280


In [28]:
# Semantic search function
def semantic_search(question, k=5):
    query_embedding = embed_query(question, client)
    results = collection.query(query_embeddings=[query_embedding], n_results=k)
    hits = []
    for i in range(len(results["documents"][0])):
        hits.append({
            "text"    : results["documents"][0][i],
            "metadata": results["metadatas"][0][i],
            "distance": results["distances"][0][i]
        })
    return hits

# answer_with_context
def answer_with_context(question, k=5):
    hits = semantic_search(question, k=k)
    context = "\n\n".join([f"[Fragment {i+1}]\n{h['text']}" for i, h in enumerate(hits)])
    system_prompt = """Eres un asistente especializado en el reglamento de Beca 18 (PRONABEC, Peru).
REGLAS ESTRICTAS:
- Responde SOLO basandote en los fragmentos de contexto proporcionados.
- Cita siempre el numero de pagina cuando aparezca en el fragmento (ej: [PAGE 12]).
- Si el contexto no contiene informacion suficiente, responde EXACTAMENTE: "El documento no contiene informacion sobre este tema."
- Nunca uses conocimiento propio. Nunca alucines.
- Responde en espanol. Se conciso y preciso."""

    full_prompt = f"""{system_prompt}

CONTEXTO RECUPERADO:
{context}

PREGUNTA: {question}

RESPUESTA:"""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=full_prompt
    )
    return {"question": question, "answer": response.text, "sources": hits}

# 5 preguntas on-topic + 1 off-topic
questions = [
    "Cuales son los requisitos academicos para postular a Beca 18 Ordinaria?",
    "Cuales son las modalidades de beca disponibles?",
    "Cuales son los beneficios economicos y montos que recibe el becario?",
    "Cuales son las obligaciones del becario durante sus estudios?",
    "En que casos se puede perder o cancelar la beca?",
    "Cual es el precio del cafe en Colombia?"
]

for q in questions:
    print(f"\n{'='*60}")
    print(f"PREGUNTA: {q}")
    print(f"{'='*60}")
    try:
        result = answer_with_context(q, k=5)
        print(f"RESPUESTA:\n{result['answer']}")
        print(f"\nFuentes: {len(result['sources'])} chunks")
    except Exception as e:
        print(f"Error: {e}")
    time.sleep(15)


PREGUNTA: Cuales son los requisitos academicos para postular a Beca 18 Ordinaria?
Error: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 1000, model: gemini-embedding-1.0\nPlease retry in 40.594804685s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/embed_content_free_tier_requests', 'quotaId': 'EmbedContentRequestsPerDayPerProjectPerModel-F

## Step 6 — Grounded Generation
The system prompt enforces strict grounding rules:
1. Answer ONLY from retrieved context — no parametric knowledge
2. Cite page numbers when available
3. Refuse with exact phrase when context is insufficient

This prevents hallucination and ensures all answers are traceable
to the source document.

In [30]:
def answer_direct(question, keywords, k=5):
    """Busca chunks por keywords sin usar embedding API"""
    hits = []
    q_words = keywords.lower().split()

    for chunk in chunks:
        score = sum(1 for w in q_words if w in chunk["text"].lower())
        if score > 0:
            hits.append((score, chunk))

    hits = sorted(hits, reverse=True)[:k]
    context = "\n\n".join([f"[Fragment {i+1}]\n{h['text']}"
                           for i, (_, h) in enumerate(hits)])

    system_prompt = """Eres un asistente especializado en el reglamento de Beca 18 (PRONABEC, Peru).
REGLAS ESTRICTAS:
- Responde SOLO basandote en los fragmentos de contexto proporcionados.
- Cita siempre el numero de pagina cuando aparezca en el fragmento (ej: [PAGE 12]).
- Si el contexto no contiene informacion suficiente, responde EXACTAMENTE: "El documento no contiene informacion sobre este tema."
- Nunca uses conocimiento propio. Nunca alucines.
- Responde en espanol. Se conciso y preciso."""

    full_prompt = f"""{system_prompt}\n\nCONTEXTO:\n{context}\n\nPREGUNTA: {question}\n\nRESPUESTA:"""

    response = client.models.generate_content(
        model="gemini-2.5-flash", contents=full_prompt)
    return {"question": question, "answer": response.text, "sources": hits}

# Preguntas con keywords específicas
qa_pairs = [
    ("Cuales son los requisitos academicos para postular a Beca 18 Ordinaria?",
     "tercio superior secundaria EBR requisito academico"),
    ("Cuales son las modalidades de beca disponibles?",
     "modalidad beca ordinaria EIB proteccion VRAEM"),
    ("Cuales son los beneficios economicos que recibe el becario?",
     "pension estudios beneficios subvencion costos academicos"),
    ("Cuales son las obligaciones del becario durante sus estudios?",
     "obligaciones becario compromiso servicio Peru"),
    ("En que casos se puede perder o cancelar la beca?",
     "cancelacion suspension perdida beca incumplimiento"),
    ("Cual es el precio del cafe en Colombia?",
     "cafe Colombia precio mercado")
]

for question, keywords in qa_pairs:
    print(f"\n{'='*60}")

In [33]:
def answer_direct(question, keywords, k=5):
    hits = []
    q_words = keywords.lower().split()

    for chunk in chunks:
        score = sum(1 for w in q_words if w in chunk["text"].lower())
        if score > 0:
            hits.append((score, chunk["text"]))

    hits = sorted(hits, key=lambda x: x[0], reverse=True)[:k]
    context = "\n\n".join([f"[Fragment {i+1}]\n{text}"
                           for i, (_, text) in enumerate(hits)])

    system_prompt = """Eres un asistente especializado en el reglamento de Beca 18 (PRONABEC, Peru).
REGLAS ESTRICTAS:
- Responde SOLO basandote en los fragmentos de contexto proporcionados.
- Cita el numero de pagina cuando aparezca (ej: [PAGE 12]).
- Si el contexto no contiene informacion suficiente, responde EXACTAMENTE: "El documento no contiene informacion sobre este tema."
- Nunca uses conocimiento propio. Responde en espanol."""

    full_prompt = f"{system_prompt}\n\nCONTEXTO:\n{context}\n\nPREGUNTA: {question}\n\nRESPUESTA:"

    response = client.models.generate_content(
        model="gemini-2.5-flash", contents=full_prompt)
    return {"answer": response.text, "sources": hits}

# Test una pregunta
result = answer_direct(
    "Cuales son las obligaciones del becario durante sus estudios?",
    "obligaciones becario compromiso servicio Peru"
)
print(result["answer"])

Las obligaciones del becario durante sus estudios son:

*   El cumplimiento del “Compromiso del Servicio al Perú” [Fragment 1, Fragment 3].
*   Las obligaciones académicas establecidas en el Reglamento y la normativa vigente del PRONABEC [PAGE 94].


In [34]:
qa_pairs = [
    ("Cuales son los requisitos academicos para postular a Beca 18 Ordinaria?",
     "tercio superior secundaria EBR requisito academico postular"),
    ("Cuales son las modalidades de beca disponibles?",
     "modalidad beca ordinaria EIB proteccion VRAEM Huallaga"),
    ("Cuales son los beneficios economicos que recibe el becario?",
     "pension estudios beneficios subvencion costos academicos"),
    ("Cuales son las obligaciones del becario durante sus estudios?",
     "obligaciones becario compromiso servicio Peru"),
    ("En que casos se puede perder o cancelar la beca?",
     "cancelacion suspension perdida beca incumplimiento renuncia"),
    ("Cual es el precio del cafe en Colombia?",
     "cafe Colombia precio mercado exportacion")
]

for question, keywords in qa_pairs:
    print(f"\n{'='*60}")
    print(f"PREGUNTA: {question}")
    print(f"{'='*60}")
    try:
        result = answer_direct(question, keywords)
        print(f"RESPUESTA:\n{result['answer']}")
    except Exception as e:
        print(f"Error: {e}")
    time.sleep(15)


PREGUNTA: Cuales son los requisitos academicos para postular a Beca 18 Ordinaria?
RESPUESTA:
El documento no contiene informacion sobre este tema.

PREGUNTA: Cuales son las modalidades de beca disponibles?
RESPUESTA:
Las modalidades de beca disponibles son las siguientes:

*   Beca 18 Ordinaria
*   Beca de Formación en Educación Intercultural Bilingüe - Beca EIB
*   Beca para Adolescentes con Protección Estatal - Beca Protección
*   Beca para Comunidades Nativas Amazónicas – Beca CNA
*   Beca para Licenciados del Servicio Militar Voluntario - Beca FF.AA.
*   Beca para pobladores del Valle de los ríos Apurímac, Ene y Mantaro - Beca VRAEM
*   Beca para pobladores residentes en el Huallaga - Beca Huallaga
*   Beca para Pueblo Afroperuano - Beca PA
*   Beca para Víctimas de la violencia habida en el país durante los años 1980 – 2000 - Beca REPARED
*   Beca Excelencia Académica para Hijos de Docentes – BEAHD

PREGUNTA: Cuales son los beneficios economicos que recibe el becario?
RESPUESTA:


In [35]:
result = answer_direct(
    "Cuales son los requisitos academicos para postular a Beca 18 Ordinaria?",
    "tercio medio superior grados secundaria calificacion rendimiento"
)
print(result["answer"])

La población objetivo del Concurso Beca 18 y Becas Especiales - Convocatoria 2026 está conformada por los jóvenes egresados de la educación secundaria con alto rendimiento académico. [PAGE 3]


In [36]:
# Step 7 — Interactive Chat Interface
import ipywidgets as widgets
from IPython.display import display, clear_output

# --- Widgets ---
question_input = widgets.Text(
    placeholder="Escribe tu pregunta sobre Beca 18...",
    layout=widgets.Layout(width="70%")
)
k_slider = widgets.IntSlider(
    value=5, min=1, max=10, step=1,
    description="k chunks:",
    layout=widgets.Layout(width="50%")
)
ask_button   = widgets.Button(description="Preguntar", button_style="primary")
clear_button = widgets.Button(description="Limpiar",   button_style="warning")
output_area  = widgets.Output()

# --- Historia del chat ---
chat_history = []

def on_ask(b):
    question = question_input.value.strip()
    if not question:
        return

    k = k_slider.value
    chat_history.append({"role": "user", "question": question})

    with output_area:
        clear_output()
        print(f"Buscando respuesta para: {question}\n")

        try:
            result = answer_direct(question, question, k=k)
            answer = result["answer"]
            sources = result["sources"]

            chat_history.append({"role": "assistant", "answer": answer})

            # Mostrar respuesta
            print(f"PREGUNTA: {question}")
            print(f"{'='*60}")
            print(f"RESPUESTA:\n{answer}")
            print()

            # Accordion con fuentes
            source_texts = []
            for i, (score, text) in enumerate(sources):
                source_texts.append(
                    f"[Fragmento {i+1}] Score: {score}\n{text[:300]}..."
                )

            accordion_children = [
                widgets.Textarea(value=t, layout=widgets.Layout(width="100%", height="100px"))
                for t in source_texts
            ]
            accordion = widgets.Accordion(children=accordion_children)
            for i in range(len(source_texts)):
                accordion.set_title(i, f"Fragmento {i+1}")
            accordion.selected_index = None

            print("Fuentes recuperadas:")
            display(accordion)

        except Exception as e:
            print(f"Error: {e}")

def on_clear(b):
    question_input.value = ""
    chat_history.clear()
    with output_area:
        clear_output()
        print("Chat limpiado.")

ask_button.on_click(on_ask)
clear_button.on_click(on_clear)

# --- Layout ---
title    = widgets.HTML("<h3>Beca 18 RAG Chatbot</h3><p>Consulta el reglamento oficial PRONABEC 2026</p>")
controls = widgets.HBox([question_input, ask_button, clear_button])
ui       = widgets.VBox([title, k_slider, controls, output_area])

display(ui)

## Conclusions
The RAG pipeline successfully retrieves relevant fragments from the 138-page
Beca 18 regulation document and generates grounded answers. The system
correctly refuses off-topic questions (e.g., coffee prices in Colombia),
demonstrating the strict system prompt prevents hallucination.

Limitation: Due to free-tier API limits (1,000 embeddings/day), only 280
chunks from key pages were indexed. A production system would index all
1,172 chunks.